# A.5 - Travelling Salesman Problem using Ant Colony Optimization
1. Ant System (AS) Algorithm
2. Max-Min Ant System (MMAS) Algorithm
3. Comparison of both algorithms

## Imports and Given Data

In [ ]:
import numpy as np
import time
import random
from itertools import permutations

In [ ]:
# Distance matrix (5 cities)
d = np.array([
    [0, 10, 12, 11, 14],
    [10,  0, 13, 15,  8],
    [12, 13,  0,  9, 14],
    [11, 15,  9,  0, 16],
    [14,  8, 14, 16,  0]
], dtype=float)

# Initial pheromone matrix (all ones)
tho_init = np.ones((5, 5), dtype=float)

n_cities = 5

print("Distance Matrix (d):")
print(d.astype(int))
print("\nInitial Pheromone Matrix (tho_init):")
print(tho_init.astype(int))

## Parameters

In [ ]:
alpha = 1.0        # Pheromone importance factor
beta = 2.0         # Heuristic (visibility) importance factor
rho = 0.5          # Evaporation rate
n_ants = 5         # Number of ants
n_iterations = 100 # Number of iterations
Q = 100.0          # Pheromone deposit constant

# Heuristic information (visibility) = 1 / distance
eta = np.zeros_like(d)
for i in range(n_cities):
    for j in range(n_cities):
        if d[i][j] != 0:
            eta[i][j] = 1.0 / d[i][j]

print("Parameters:")
print(f"  Alpha (pheromone weight)  : {alpha}")
print(f"  Beta  (heuristic weight)  : {beta}")
print(f"  Rho   (evaporation rate)  : {rho}")
print(f"  Q     (deposit constant)  : {Q}")
print(f"  Number of ants            : {n_ants}")
print(f"  Number of iterations      : {n_iterations}")
print(f"\nHeuristic Matrix (eta = 1/d):")
print(np.round(eta, 4))

## Helper Functions

In [ ]:
def calculate_tour_length(tour, dist_matrix):
    """Calculate total distance of a tour."""
    length = 0
    for i in range(len(tour) - 1):
        length += dist_matrix[tour[i]][tour[i + 1]]
    length += dist_matrix[tour[-1]][tour[0]]  # Return to start
    return length


def select_next_city(pheromone, heuristic, current, visited, alpha, beta):
    """Select next city using roulette wheel selection based on transition probabilities."""
    n = len(pheromone)
    probabilities = np.zeros(n)

    for j in range(n):
        if j not in visited:
            probabilities[j] = (pheromone[current][j] ** alpha) * (heuristic[current][j] ** beta)

    total = probabilities.sum()
    if total == 0:
        unvisited = [j for j in range(n) if j not in visited]
        return random.choice(unvisited)

    probabilities /= total

    # Roulette wheel selection
    r = random.random()
    cumulative = 0
    for j in range(n):
        cumulative += probabilities[j]
        if r <= cumulative:
            return j

    unvisited = [j for j in range(n) if j not in visited]
    return unvisited[-1]


def construct_solution(pheromone, heuristic, n_cities, alpha, beta):
    """Construct a tour for one ant."""
    start = random.randint(0, n_cities - 1)
    tour = [start]
    visited = {start}

    for _ in range(n_cities - 1):
        next_city = select_next_city(pheromone, heuristic, tour[-1], visited, alpha, beta)
        tour.append(next_city)
        visited.add(next_city)

    return tour


def format_tour(tour):
    """Format tour as 1-indexed city names."""
    return " -> ".join([str(c + 1) for c in tour]) + " -> " + str(tour[0] + 1)

print("Helper functions defined successfully.")

---
## 1. Ant System (AS) Algorithm

In the standard Ant System, **all ants** deposit pheromone on their respective paths after each iteration. The pheromone update rule is:

- **Evaporation**: $\tau_{ij} = (1 - \rho) \cdot \tau_{ij}$
- **Deposit**: $\tau_{ij} = \tau_{ij} + \sum_{k=1}^{m} \Delta\tau_{ij}^{k}$ where $\Delta\tau_{ij}^{k} = Q / L_k$ if ant $k$ uses edge $(i,j)$

In [ ]:
def ant_system(dist_matrix, pheromone_init, eta, n_ants, n_iterations,
               alpha, beta, rho, Q):
    """
    Standard Ant System Algorithm.
    Pheromone update: All ants deposit pheromone on their paths.
    """
    n = len(dist_matrix)
    pheromone = pheromone_init.copy()

    best_tour = None
    best_length = float('inf')

    iteration_best_lengths = []

    for iteration in range(n_iterations):
        all_tours = []
        all_lengths = []

        # Step 1: Construct solutions for all ants
        for ant in range(n_ants):
            tour = construct_solution(pheromone, eta, n, alpha, beta)
            length = calculate_tour_length(tour, dist_matrix)
            all_tours.append(tour)
            all_lengths.append(length)

            if length < best_length:
                best_length = length
                best_tour = tour[:]

        iteration_best_lengths.append(min(all_lengths))

        # Step 2: Pheromone evaporation
        pheromone *= (1 - rho)

        # Step 3: Pheromone deposit (ALL ants contribute)
        for ant in range(n_ants):
            tour = all_tours[ant]
            length = all_lengths[ant]
            delta = Q / length

            for i in range(n - 1):
                pheromone[tour[i]][tour[i + 1]] += delta
                pheromone[tour[i + 1]][tour[i]] += delta  # Symmetric
            # Close the tour
            pheromone[tour[-1]][tour[0]] += delta
            pheromone[tour[0]][tour[-1]] += delta

    return best_tour, best_length, pheromone, iteration_best_lengths

print("Ant System function defined.")

### Run Ant System

In [ ]:
random.seed(42)
np.random.seed(42)

start_time = time.time()
as_tour, as_length, as_pheromone, as_iter_best = ant_system(
    d, tho_init, eta, n_ants, n_iterations, alpha, beta, rho, Q
)
as_time = time.time() - start_time

print("=" * 60)
print("  ANT SYSTEM (AS) - RESULTS")
print("=" * 60)
print(f"\n  Best Tour Found     : {format_tour(as_tour)}")
print(f"  Best Tour Length    : {as_length}")
print(f"  Execution Time      : {as_time:.6f} seconds")
print(f"\n  Final Pheromone Matrix (AS):")
print(np.round(as_pheromone, 3))

---
## 2. Max-Min Ant System (MMAS) Algorithm

Key differences from standard AS:
- Only the **best ant** (iteration-best or global-best) deposits pheromone
- Pheromone values are **clamped** within $[\tau_{min}, \tau_{max}]$
- Pheromone trails are **initialized to** $\tau_{max}$

In [ ]:
def max_min_ant_system(dist_matrix, pheromone_init, eta, n_ants, n_iterations,
                       alpha, beta, rho, Q, tau_max=5.0, tau_min=0.1):
    """
    Max-Min Ant System (MMAS) Algorithm.
    Key differences from standard AS:
      - Only the BEST ant deposits pheromone.
      - Pheromone values are clamped within [tau_min, tau_max].
      - Pheromone trails are initialized to tau_max.
    """
    n = len(dist_matrix)
    pheromone = np.full((n, n), tau_max)  # Initialize to tau_max

    best_tour = None
    best_length = float('inf')

    iteration_best_lengths = []

    for iteration in range(n_iterations):
        all_tours = []
        all_lengths = []

        # Step 1: Construct solutions for all ants
        for ant in range(n_ants):
            tour = construct_solution(pheromone, eta, n, alpha, beta)
            length = calculate_tour_length(tour, dist_matrix)
            all_tours.append(tour)
            all_lengths.append(length)

            if length < best_length:
                best_length = length
                best_tour = tour[:]

        iteration_best_lengths.append(min(all_lengths))

        # Find the iteration-best ant
        iter_best_idx = np.argmin(all_lengths)
        iter_best_tour = all_tours[iter_best_idx]
        iter_best_length = all_lengths[iter_best_idx]

        # Step 2: Pheromone evaporation
        pheromone *= (1 - rho)

        # Step 3: Pheromone deposit (ONLY best ant deposits)
        if iteration < n_iterations // 2:
            deposit_tour = iter_best_tour
            deposit_length = iter_best_length
        else:
            deposit_tour = best_tour
            deposit_length = best_length

        delta = Q / deposit_length

        for i in range(n - 1):
            pheromone[deposit_tour[i]][deposit_tour[i + 1]] += delta
            pheromone[deposit_tour[i + 1]][deposit_tour[i]] += delta
        pheromone[deposit_tour[-1]][deposit_tour[0]] += delta
        pheromone[deposit_tour[0]][deposit_tour[-1]] += delta

        # Step 4: Clamp pheromone within [tau_min, tau_max]
        pheromone = np.clip(pheromone, tau_min, tau_max)

    return best_tour, best_length, pheromone, iteration_best_lengths

print("Max-Min Ant System function defined.")

### Run Max-Min Ant System

In [ ]:
random.seed(42)
np.random.seed(42)

start_time = time.time()
mmas_tour, mmas_length, mmas_pheromone, mmas_iter_best = max_min_ant_system(
    d, tho_init, eta, n_ants, n_iterations, alpha, beta, rho, Q,
    tau_max=5.0, tau_min=0.1
)
mmas_time = time.time() - start_time

print("=" * 60)
print("  MAX-MIN ANT SYSTEM (MMAS) - RESULTS")
print("=" * 60)
print(f"\n  Best Tour Found     : {format_tour(mmas_tour)}")
print(f"  Best Tour Length    : {mmas_length}")
print(f"  Execution Time      : {mmas_time:.6f} seconds")
print(f"\n  Final Pheromone Matrix (MMAS):")
print(np.round(mmas_pheromone, 3))

---
## 3. Comparison of AS vs MMAS

In [ ]:
print("=" * 60)
print("  COMPARISON TABLE: AS vs MMAS")
print("=" * 60)
print(f"\n{'Criterion':<25} {'Ant System':<20} {'Max-Min AS':<20}")
print("-" * 65)
print(f"{'Best Tour Length':<25} {as_length:<20} {mmas_length:<20}")
print(f"{'Best Tour':<25} {format_tour(as_tour):<20} {format_tour(mmas_tour):<20}")
print(f"{'Execution Time (sec)':<25} {as_time:<20.6f} {mmas_time:<20.6f}")
print(f"{'Convergence (iter 10)':<25} {as_iter_best[9]:<20} {mmas_iter_best[9]:<20}")
print(f"{'Convergence (iter 50)':<25} {as_iter_best[49]:<20} {mmas_iter_best[49]:<20}")
print(f"{'Convergence (iter 100)':<25} {as_iter_best[99]:<20} {mmas_iter_best[99]:<20}")
print("-" * 65)

if mmas_length < as_length:
    print("\n>> MMAS found a BETTER solution than AS.")
elif mmas_length == as_length:
    print("\n>> Both algorithms found the SAME optimal solution.")
else:
    print("\n>> AS found a BETTER solution than MMAS in this run.")

### Convergence Plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(range(1, n_iterations + 1), as_iter_best, label='Ant System (AS)', linewidth=2)
plt.plot(range(1, n_iterations + 1), mmas_iter_best, label='Max-Min AS (MMAS)', linewidth=2, linestyle='--')
plt.xlabel('Iteration')
plt.ylabel('Best Tour Length')
plt.title('Convergence Comparison: AS vs MMAS')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Brute Force Verification

In [ ]:
best_bf_length = float('inf')
best_bf_tour = None
for perm in permutations(range(n_cities)):
    length = calculate_tour_length(list(perm), d)
    if length < best_bf_length:
        best_bf_length = length
        best_bf_tour = list(perm)

print("=" * 60)
print("  BRUTE FORCE VERIFICATION")
print("=" * 60)
print(f"\n  Optimal Tour        : {format_tour(best_bf_tour)}")
print(f"  Optimal Tour Length : {best_bf_length}")
print(f"  AS  found optimal?  : {'Yes' if as_length == best_bf_length else 'No (gap: ' + str(as_length - best_bf_length) + ')'}")
print(f"  MMAS found optimal? : {'Yes' if mmas_length == best_bf_length else 'No (gap: ' + str(mmas_length - best_bf_length) + ')'}")

### Time and Complexity Analysis

In [ ]:
print("=" * 60)
print("  TIME & COMPLEXITY ANALYSIS")
print("=" * 60)
print(f"""
TIME COMPLEXITY:
  * AS  Algorithm : O(n_iterations x n_ants x n_cities^2)
    - All ants deposit pheromone => O(n_ants x n_cities) per update
  * MMAS Algorithm: O(n_iterations x n_ants x n_cities^2)
    - Only best ant deposits => O(n_cities) per update
    - Additional clipping step => O(n_cities^2) per iteration
    - Same asymptotic complexity, less pheromone update work

SPACE COMPLEXITY:
  * Both: O(n_cities^2) for pheromone and distance matrices

KEY DIFFERENCES:
  * AS: ALL ants update pheromone => can lead to stagnation
  * MMAS: Only BEST ant updates + pheromone bounds [tau_min, tau_max]:
    => Avoids premature convergence (capped at tau_max)
    => Maintains exploration (floored at tau_min)
  * MMAS generally better for larger problem instances

MEASURED EXECUTION TIMES:
  * AS  : {as_time:.6f} seconds
  * MMAS: {mmas_time:.6f} seconds
""")